In [ ]:
import pandas as pd
import numpy as np
import ast
import re

In [ ]:
movies = pd.read_csv('movies_metadata.csv', low_memory=False)
credits = pd.read_csv('credits.csv')
rotten = pd.read_csv('rotten_tomatoes_movies.csv')

In [ ]:
movies = movies[['id', 'title', 'release_date', 'budget', 'revenue', 'runtime', 'genres']]

In [ ]:
movies['budget'] = pd.to_numeric(movies['budget'], errors='coerce')
movies['revenue'] = pd.to_numeric(movies['revenue'], errors='coerce')
movies['runtime'] = pd.to_numeric(movies['runtime'], errors='coerce')

In [ ]:
movies['year'] = pd.to_datetime(movies['release_date'], errors='coerce').dt.year

In [ ]:
def clean_title(title):
    if pd.isna(title):
        return ""
    title = title.lower()
    title = re.sub(r'[^a-z0-9\s]', '', title)
    title = re.sub(r'\s+', ' ', title).strip()
    return title
movies['title_clean'] = movies['title'].apply(clean_title)

In [ ]:
def parse_genres(x):
    try:
        genres = ast.literal_eval(x)
        return [g['name'] for g in genres]
    except:
        return []
movies['genres'] = movies['genres'].apply(parse_genres)

In [ ]:
credits['id'] = credits['id'].astype(str)
movies['id'] = movies['id'].astype(str)

In [ ]:
def get_director(crew):
    try:
        crew = ast.literal_eval(crew)
        for person in crew:
            if person['job'] == 'Director':
                return person['name']
    except:
        return None
    return None
credits['director'] = credits['crew'].apply(get_director)

In [ ]:
def get_cast(cast):
    try:
        cast = ast.literal_eval(cast)
        return [actor['name'] for actor in cast[:5]]
    except:
        return []
credits['cast'] = credits['cast'].apply(get_cast)

In [ ]:
credits = credits[['id', 'director', 'cast']]

In [ ]:
df = movies.merge(credits, on='id', how='inner')

In [ ]:
df.shape

(45538, 11)

In [ ]:
rotten = rotten[[
    'movie_title',
    'tomatometer_rating',
    'audience_rating'
]]

In [ ]:
rotten['title_clean'] = rotten['movie_title'].apply(clean_title)

In [ ]:
rotten['tomatometer_rating'] = pd.to_numeric(rotten['tomatometer_rating'], errors='coerce')
rotten['audience_rating'] = pd.to_numeric(rotten['audience_rating'], errors='coerce')

In [ ]:
df = df.merge(
    rotten,
    on='title_clean',
    how='left'
)

In [ ]:
df = df.drop_duplicates(subset=['title_clean', 'year'])

In [ ]:
df.shape

(45368, 14)

In [ ]:
df.head()

,id,title,release_date,budget,revenue,runtime,genres,year,title_clean,director,cast,movie_title,tomatometer_rating,audience_rating
0,862,Toy Story,1995-10-30,30000000.0,373554033.0,81.0,"[Animation, Comedy, Family]",1995.0,toy story,John Lasseter,"[Tom Hanks, Tim Allen, Don Rickles, Jim Varney...",Toy Story,100.0,92.0
1,8844,Jumanji,1995-12-15,65000000.0,262797249.0,104.0,"[Adventure, Fantasy, Family]",1995.0,jumanji,Joe Johnston,"[Robin Williams, Jonathan Hyde, Kirsten Dunst,...",Jumanji,55.0,62.0
2,15602,Grumpier Old Men,1995-12-22,0.0,0.0,101.0,"[Romance, Comedy]",1995.0,grumpier old men,Howard Deutch,"[Walter Matthau, Jack Lemmon, Ann-Margret, Sop...",Grumpier Old Men,17.0,62.0
3,31357,Waiting to Exhale,1995-12-22,16000000.0,81452156.0,127.0,"[Comedy, Drama, Romance]",1995.0,waiting to exhale,Forest Whitaker,"[Whitney Houston, Angela Bassett, Loretta Devi...",Waiting to Exhale,56.0,79.0
4,11862,Father of the Bride Part II,1995-02-10,0.0,76578911.0,106.0,[Comedy],1995.0,father of the bride part ii,Charles Shyer,"[Steve Martin, Diane Keaton, Martin Short, Kim...",Father of the Bride: Part II,48.0,60.0


In [ ]:
df = df.drop(columns=['movie_title'])

In [ ]:
df = df[[
    'id', 'title', 'year', 'budget', 'revenue', 'runtime',
    'genres', 'director', 'cast',
    'tomatometer_rating', 'audience_rating'
]]

In [ ]:
df.head()

,id,title,year,budget,revenue,runtime,genres,director,cast,tomatometer_rating,audience_rating
0,862,Toy Story,1995.0,30000000.0,373554033.0,81.0,"[Animation, Comedy, Family]",John Lasseter,"[Tom Hanks, Tim Allen, Don Rickles, Jim Varney...",100.0,92.0
1,8844,Jumanji,1995.0,65000000.0,262797249.0,104.0,"[Adventure, Fantasy, Family]",Joe Johnston,"[Robin Williams, Jonathan Hyde, Kirsten Dunst,...",55.0,62.0
2,15602,Grumpier Old Men,1995.0,0.0,0.0,101.0,"[Romance, Comedy]",Howard Deutch,"[Walter Matthau, Jack Lemmon, Ann-Margret, Sop...",17.0,62.0
3,31357,Waiting to Exhale,1995.0,16000000.0,81452156.0,127.0,"[Comedy, Drama, Romance]",Forest Whitaker,"[Whitney Houston, Angela Bassett, Loretta Devi...",56.0,79.0
4,11862,Father of the Bride Part II,1995.0,0.0,76578911.0,106.0,[Comedy],Charles Shyer,"[Steve Martin, Diane Keaton, Martin Short, Kim...",48.0,60.0


In [ ]:
df.shape

(45368, 11)

In [ ]:
acting = pd.read_csv('oscardata_acting.csv')
director = pd.read_csv('oscardata_bestdirector.csv')
picture = pd.read_csv('oscardata_bestpicture.csv')

In [ ]:
oscar = pd.concat([acting, director, picture])

In [ ]:
oscar = oscar[['Film', 'Year', 'Winner', 'Category']]

In [ ]:
oscar['title_clean'] = oscar['Film'].apply(clean_title)

In [ ]:
oscar['nominated'] = 1
oscar['won'] = oscar['Winner']

In [ ]:
oscar_grouped = oscar.groupby(['title_clean', 'Year']).agg({
    'nominated': 'sum',
    'won': 'sum'
}).reset_index()

In [ ]:
oscar['best_picture_nom'] = (oscar['Category'] == 'Picture').astype(int)

best_picture = oscar.groupby(['title_clean', 'Year'])['best_picture_nom'].max().reset_index()

oscar_final = oscar_grouped.merge(best_picture, on=['title_clean', 'Year'])

In [ ]:
df['title_clean'] = df['title'].apply(clean_title)

In [ ]:
df = df.merge(
    oscar_final,
    on='title_clean',
    how='left'
)

In [ ]:
df['nominated'] = df['nominated'].fillna(0)
df['won'] = df['won'].fillna(0)
df['best_picture_nom'] = df['best_picture_nom'].fillna(0)

In [ ]:
df.shape

(45370, 16)

In [ ]:
df.head()

,id,title,year,budget,revenue,runtime,genres,director,cast,tomatometer_rating,audience_rating,title_clean,Year,nominated,won,best_picture_nom
0,862,Toy Story,1995.0,30000000.0,373554033.0,81.0,"[Animation, Comedy, Family]",John Lasseter,"[Tom Hanks, Tim Allen, Don Rickles, Jim Varney...",100.0,92.0,toy story,NaN,0.0,0.0,0.0
1,8844,Jumanji,1995.0,65000000.0,262797249.0,104.0,"[Adventure, Fantasy, Family]",Joe Johnston,"[Robin Williams, Jonathan Hyde, Kirsten Dunst,...",55.0,62.0,jumanji,NaN,0.0,0.0,0.0
2,15602,Grumpier Old Men,1995.0,0.0,0.0,101.0,"[Romance, Comedy]",Howard Deutch,"[Walter Matthau, Jack Lemmon, Ann-Margret, Sop...",17.0,62.0,grumpier old men,NaN,0.0,0.0,0.0
3,31357,Waiting to Exhale,1995.0,16000000.0,81452156.0,127.0,"[Comedy, Drama, Romance]",Forest Whitaker,"[Whitney Houston, Angela Bassett, Loretta Devi...",56.0,79.0,waiting to exhale,NaN,0.0,0.0,0.0
4,11862,Father of the Bride Part II,1995.0,0.0,76578911.0,106.0,[Comedy],Charles Shyer,"[Steve Martin, Diane Keaton, Martin Short, Kim...",48.0,60.0,father of the bride part ii,NaN,0.0,0.0,0.0


In [ ]:
df['best_picture_nom'].value_counts()

,count
best_picture_nom,
0.0,44975
1.0,395


In [ ]:
df['won'].value_counts()

,count
won,
0.0,45080
1.0,203
2.0,58
3.0,23
4.0,6


In [ ]:
df.columns

Index(['id', 'title', 'year', 'budget', 'revenue', 'runtime', 'genres',
       'director', 'cast', 'tomatometer_rating', 'audience_rating',
       'title_clean', 'Year', 'nominated', 'won', 'best_picture_nom'],
      dtype='object')

In [ ]:
df = df.drop(columns=['title_clean', 'Year'])

In [ ]:
df.columns

Index(['id', 'title', 'year', 'budget', 'revenue', 'runtime', 'genres',
       'director', 'cast', 'tomatometer_rating', 'audience_rating',
       'nominated', 'won', 'best_picture_nom'],
      dtype='object')

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45370 entries, 0 to 45369
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  45370 non-null  object 
 1   title               45369 non-null  object 
 2   year                45286 non-null  float64
 3   budget              45370 non-null  float64
 4   revenue             45369 non-null  float64
 5   runtime             45118 non-null  float64
 6   genres              45370 non-null  object 
 7   director            44487 non-null  object 
 8   cast                45370 non-null  object 
 9   tomatometer_rating  14897 non-null  float64
 10  audience_rating     14840 non-null  float64
 11  nominated           45370 non-null  float64
 12  won                 45370 non-null  float64
 13  best_picture_nom    45370 non-null  float64
dtypes: float64(9), object(5)
memory usage: 4.8+ MB


In [ ]:
df['year'] = df['year'].fillna(df['year'].median()).astype(int)

df['nominated'] = df['nominated'].astype(int)
df['won'] = df['won'].astype(int)
df['best_picture_nom'] = df['best_picture_nom'].astype(int)

In [ ]:
df.isnull().sum()

,0
id,0
title,1
year,0
budget,0
revenue,1
runtime,252
genres,0
director,883
cast,0
tomatometer_rating,30473


In [ ]:
df = df.dropna(subset=['title'])
df['revenue'] = df['revenue'].fillna(df['revenue'].median())
df['runtime'] = df['runtime'].fillna(df['runtime'].median())
df['director'] = df['director'].fillna('unknown')
df['tomatometer_rating'] = df['tomatometer_rating'].fillna(df['tomatometer_rating'].median())
df['audience_rating'] = df['audience_rating'].fillna(df['audience_rating'].median())

In [ ]:
df.isnull().sum()

,0
id,0
title,0
year,0
budget,0
revenue,0
runtime,0
genres,0
director,0
cast,0
tomatometer_rating,0


In [ ]:
df

,id,title,year,budget,revenue,runtime,genres,director,cast,tomatometer_rating,audience_rating,nominated,won,best_picture_nom
0,862,Toy Story,1995,30000000.0,373554033.0,81.0,"[Animation, Comedy, Family]",John Lasseter,"[Tom Hanks, Tim Allen, Don Rickles, Jim Varney...",100.0,92.0,0,0,0
1,8844,Jumanji,1995,65000000.0,262797249.0,104.0,"[Adventure, Fantasy, Family]",Joe Johnston,"[Robin Williams, Jonathan Hyde, Kirsten Dunst,...",55.0,62.0,0,0,0
2,15602,Grumpier Old Men,1995,0.0,0.0,101.0,"[Romance, Comedy]",Howard Deutch,"[Walter Matthau, Jack Lemmon, Ann-Margret, Sop...",17.0,62.0,0,0,0
3,31357,Waiting to Exhale,1995,16000000.0,81452156.0,127.0,"[Comedy, Drama, Romance]",Forest Whitaker,"[Whitney Houston, Angela Bassett, Loretta Devi...",56.0,79.0,0,0,0
4,11862,Father of the Bride Part II,1995,0.0,76578911.0,106.0,[Comedy],Charles Shyer,"[Steve Martin, Diane Keaton, Martin Short, Kim...",48.0,60.0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45365,439050,Subdue,2001,0.0,0.0,90.0,"[Drama, Family]",Hamid Nematollah,"[Leila Hatami, Kourosh Tahami, Elham Korda]",65.0,62.0,0,0,0
45366,111109,Century of Birthing,2011,0.0,0.0,360.0,[Drama],Lav Diaz,"[Angel Aquino, Perry Dizon, Hazel Orencio, Joe...",65.0,62.0,0,0,0
45367,67758,Betrayal,2003,0.0,0.0,90.0,"[Action, Drama, Thriller]",Mark L. Lester,"[Erika Eleniak, Adam Baldwin, Julie du Page, J...",65.0,62.0,0,0,0
45368,227506,Satan Triumphant,1917,0.0,0.0,87.0,[],Yakov Protazanov,"[Iwan Mosschuchin, Nathalie Lissenko, Pavel Pa...",65.0,62.0,0,0,0


In [ ]:
df[['budget', 'revenue', 'runtime', 'tomatometer_rating', 'audience_rating']].describe()

,budget,revenue,runtime,tomatometer_rating,audience_rating
count,4.536900e+04,4.536900e+04,45369.000000,45369.000000,45369.000000
mean,4.230902e+06,1.123383e+07,94.123168,63.291058,61.412749
std,1.744028e+07,6.440397e+07,38.292659,16.591955,11.680651
min,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000
25%,0.000000e+00,0.000000e+00,85.000000,65.000000,62.000000
50%,0.000000e+00,0.000000e+00,95.000000,65.000000,62.000000
75%,0.000000e+00,0.000000e+00,107.000000,65.000000,62.000000
max,3.800000e+08,2.787965e+09,1256.000000,100.000000,100.000000


In [ ]:
df['year'].describe()

,year
count,45369.000000
mean,1991.882144
std,24.038945
min,1874.000000
25%,1978.000000
50%,2001.000000
75%,2010.000000
max,2020.000000


In [ ]:
df['genres'].head(10)

,genres
0,"[Animation, Comedy, Family]"
1,"[Adventure, Fantasy, Family]"
2,"[Romance, Comedy]"
3,"[Comedy, Drama, Romance]"
4,[Comedy]
5,"[Action, Crime, Drama, Thriller]"
6,"[Comedy, Romance]"
7,"[Action, Adventure, Drama, Family]"
8,"[Action, Adventure, Thriller]"
9,"[Adventure, Action, Thriller]"


In [ ]:
df['director'].value_counts().head(10)

,count
director,
unknown,883
John Ford,66
Michael Curtiz,65
Werner Herzog,54
Alfred Hitchcock,53
Georges Méliès,51
Woody Allen,49
Jean-Luc Godard,47
Sidney Lumet,46


In [ ]:
df['cast'].head()

,cast
0,"[Tom Hanks, Tim Allen, Don Rickles, Jim Varney..."
1,"[Robin Williams, Jonathan Hyde, Kirsten Dunst,..."
2,"[Walter Matthau, Jack Lemmon, Ann-Margret, Sop..."
3,"[Whitney Houston, Angela Bassett, Loretta Devi..."
4,"[Steve Martin, Diane Keaton, Martin Short, Kim..."


In [ ]:
df['tomatometer_rating'].describe()


,tomatometer_rating
count,45369.000000
mean,63.291058
std,16.591955
min,0.000000
25%,65.000000
50%,65.000000
75%,65.000000
max,100.000000


In [ ]:
df['audience_rating'].describe()

,audience_rating
count,45369.000000
mean,61.412749
std,11.680651
min,0.000000
25%,62.000000
50%,62.000000
75%,62.000000
max,100.000000


In [ ]:
df['best_picture_nom'].value_counts()

,count
best_picture_nom,
0,44974
1,395


In [ ]:
df

,id,title,year,budget,revenue,runtime,genres,director,cast,tomatometer_rating,audience_rating,nominated,won,best_picture_nom
0,862,Toy Story,1995,30000000.0,373554033.0,81.0,"[Animation, Comedy, Family]",John Lasseter,"[Tom Hanks, Tim Allen, Don Rickles, Jim Varney...",100.0,92.0,0,0,0
1,8844,Jumanji,1995,65000000.0,262797249.0,104.0,"[Adventure, Fantasy, Family]",Joe Johnston,"[Robin Williams, Jonathan Hyde, Kirsten Dunst,...",55.0,62.0,0,0,0
2,15602,Grumpier Old Men,1995,0.0,0.0,101.0,"[Romance, Comedy]",Howard Deutch,"[Walter Matthau, Jack Lemmon, Ann-Margret, Sop...",17.0,62.0,0,0,0
3,31357,Waiting to Exhale,1995,16000000.0,81452156.0,127.0,"[Comedy, Drama, Romance]",Forest Whitaker,"[Whitney Houston, Angela Bassett, Loretta Devi...",56.0,79.0,0,0,0
4,11862,Father of the Bride Part II,1995,0.0,76578911.0,106.0,[Comedy],Charles Shyer,"[Steve Martin, Diane Keaton, Martin Short, Kim...",48.0,60.0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45365,439050,Subdue,2001,0.0,0.0,90.0,"[Drama, Family]",Hamid Nematollah,"[Leila Hatami, Kourosh Tahami, Elham Korda]",65.0,62.0,0,0,0
45366,111109,Century of Birthing,2011,0.0,0.0,360.0,[Drama],Lav Diaz,"[Angel Aquino, Perry Dizon, Hazel Orencio, Joe...",65.0,62.0,0,0,0
45367,67758,Betrayal,2003,0.0,0.0,90.0,"[Action, Drama, Thriller]",Mark L. Lester,"[Erika Eleniak, Adam Baldwin, Julie du Page, J...",65.0,62.0,0,0,0
45368,227506,Satan Triumphant,1917,0.0,0.0,87.0,[],Yakov Protazanov,"[Iwan Mosschuchin, Nathalie Lissenko, Pavel Pa...",65.0,62.0,0,0,0


In [ ]:
df['nominated'].value_counts()

,count
nominated,
0,44364
1,488
2,217
3,132
4,99
5,43
6,23
7,3


In [ ]:
df['won'].value_counts()

,count
won,
0,45079
1,203
2,58
3,23
4,6


In [ ]:
df[df['nominated'] >= 6][['title', 'year', 'nominated']].head(10)

,title,year,nominated
834,The Godfather,1972,6
1052,Bonnie and Clyde,1967,7
1178,The Godfather: Part II,1974,7
1844,Rocky,1976,6
1845,Kramer vs. Kramer,1979,6
1848,Terms of Endearment,1983,6
2022,Who's Afraid of Virginia Woolf?,1966,6
2812,Reds,1981,6
3034,The Last Picture Show,1971,6
3330,Guess Who's Coming to Dinner,1967,6


In [ ]:
df.to_csv('ML_dataset.csv', index=False)